# 2 - Accessing related items through relationships

Notebook 1 showed that every resource carries a `relationships` block. Here we use it to walk from a
*Project* to the things it contains: its members, data files, models and so on.

**What you will learn**

* how `relationships` is structured and how it maps to URLs
* how to follow a relationship and fetch the related resources
* how to collect the results into a table
* why only *visible* items appear, and what that means for anonymous access

As in notebook 1 we read public data from the FAIRDOMHub, so no login is needed.

## 2.1 Setup

The same imports, `base_url`, session and helper as in notebook 1. Every notebook repeats this block so
that it can be run on its own.

In [1]:
import json

import requests
import pandas as pd
from pandas import json_normalize

base_url = "https://fairdomhub.org"

session = requests.Session()
session.headers.update({"Accept": "application/json"})


def json_for_resource(resource_type, resource_id):
    '''GET {base_url}/{resource_type}/{resource_id} and return the parsed JSON.'''
    url = f"{base_url}/{resource_type}/{resource_id}"
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

## 2.2 Fetch a project and look at its relationships

We fetch [project 60](https://fairdomhub.org/projects/60) and print only the `relationships` block,
because that is what we are interested in here.

Relationships are grouped by type (`people`, `institutions`, `data_files`, ...). Each group has a
`data` list, and each entry in it is a **reference**: just a `type` and an `id`. That is exactly what
`json_for_resource` needs, so `{'id': '18', 'type': 'institutions'}` means
https://fairdomhub.org/institutions/18.

**Only visible items are listed.** Related items that you are not allowed to see because of their
sharing policy are silently left out. Anonymous requests, like ours, see only public items; with an
API token (notebooks 4 to 6) the same call can return more.

In [2]:
project_id = 60

project = json_for_resource("projects", project_id)

print("Project:", project["data"]["attributes"]["title"])
print()
print(json.dumps(project["data"]["relationships"], indent=2))

Project: Systems toxicology of Atlantic cod

{
  "project_administrators": {
    "data": [
      {
        "id": "847",
        "type": "people"
      },
      {
        "id": "713",
        "type": "people"
      },
      {
        "id": "853",
        "type": "people"
      },
      {
        "id": "848",
        "type": "people"
      },
      {
        "id": "849",
        "type": "people"
      },
      {
        "id": "852",
        "type": "people"
      },
      {
        "id": "1125",
        "type": "people"
      }
    ]
  },
  "pals": {
    "data": []
  },
  "asset_housekeepers": {
    "data": []
  },
  "asset_gatekeepers": {
    "data": []
  },
  "organisms": {
    "data": [
      {
        "id": "1933753720",
        "type": "organisms"
      }
    ]
  },
  "human_diseases": {
    "data": []
  },
  "people": {
    "data": [
      {
        "id": "697",
        "type": "people"
      },
      {
        "id": "713",
        "type": "people"
      },
      {
        "id": "8

That is a long list. A summary with one line per relationship type is easier to read:

In [3]:
for name, rel in project["data"]["relationships"].items():
    print(f"{name:25} {len(rel['data']):4} item(s)")

project_administrators       7 item(s)
pals                         0 item(s)
asset_housekeepers           0 item(s)
asset_gatekeepers            0 item(s)
organisms                    1 item(s)
human_diseases               0 item(s)
people                      26 item(s)
institutions                 4 item(s)
programmes                   1 item(s)
investigations               0 item(s)
studies                      7 item(s)
assays                      34 item(s)
data_files                  57 item(s)
file_templates               0 item(s)
placeholders                 0 item(s)
models                       3 item(s)
sops                         0 item(s)
publications                11 item(s)
presentations                0 item(s)
events                       0 item(s)
documents                    4 item(s)
workflows                    0 item(s)
collections                  0 item(s)


## 2.3 Follow a relationship

The references only give `type` and `id`. To learn anything else, e.g. the title, we fetch each related
resource. The function below does that for one relationship type and collects `id`, `title` and the
URL of each item.

Note that this sends **one request per related item**. For a handful of data files that is fine; for
hundreds of items it becomes slow, and section 2.5 shows a cheaper alternative.

In [4]:
def related_items(resource, relationship):
    '''Fetch every item of one relationship and return a list of small dicts.'''
    items = []
    for ref in resource["data"]["relationships"][relationship]["data"]:
        item = json_for_resource(ref["type"], ref["id"])
        items.append({
            "id": item["data"]["id"],
            "type": item["data"]["type"],
            "title": item["data"]["attributes"]["title"],
            "url": f"{base_url}/{ref['type']}/{ref['id']}",
        })
    return items


data_files = related_items(project, "data_files")

print(len(data_files), "data files found")

57 data files found


## 2.4 Show the result as a table

`json_normalize` turns the list of dictionaries into a `DataFrame`, one row per data file. From here
you can sort, filter or export to CSV with the usual pandas tools.

In [5]:
table = json_normalize(data_files)

table.head(15)

,id,type,title,url
0,1880,data_files,Samples for dCod,https://fairdomhub.org/data_files/1880
1,1883,data_files,Library preparation,https://fairdomhub.org/data_files/1883
2,1923,data_files,FASTQ Paired,https://fairdomhub.org/data_files/1923
3,1924,data_files,Bam files,https://fairdomhub.org/data_files/1924
4,1925,data_files,Count files,https://fairdomhub.org/data_files/1925
5,2249,data_files,Kollevåg biometrics,https://fairdomhub.org/data_files/2249
6,2250,data_files,PAH metabolites in bile Kollevåg,https://fairdomhub.org/data_files/2250
7,2251,data_files,Steroid hormones plasma Kollevåg,https://fairdomhub.org/data_files/2251
8,2255,data_files,Enzyme activity Gst+Cat_Kollevåg,https://fairdomhub.org/data_files/2255
9,2256,data_files,In vivo-1 biometrics,https://fairdomhub.org/data_files/2256


The same function works for any relationship type, because every SEEK resource has a `title`.
Here are the models of the same project:

In [6]:
json_normalize(related_items(project, "models"))

,id,type,title,url
0,756,models,Draft metabolic reconstruction model of Atlant...,https://fairdomhub.org/models/756
1,757,models,Draft metabolic reconstruction model of Atlant...,https://fairdomhub.org/models/757
2,758,models,Draft metabolic reconstruction model of Atlant...,https://fairdomhub.org/models/758


## 2.5 The cheaper way: a nested listing

Fetching items one by one costs one request each: 57 requests for the table above. SEEK also offers
**listing endpoints** that return `id`, `type` and `title` for many items at once. The response is a
JSON:API *collection*: `data` is a list instead of a single object.

Two forms exist:

* `/data_files` lists **every** data file you are allowed to see. On the FAIRDOMHub that is several
  thousand rows in one response, so it is rarely what you want.
* `/projects/60/data_files` lists only the data files **of that project**, in one request. This nested
  form exists for every relationship shown in section 2.2 (`/projects/60/people`,
  `/projects/60/models`, `/investigations/12/studies`, ...).

The listing only carries `title`; for other attributes you still fetch the individual resource.

In [7]:
response = session.get(f"{base_url}/projects/{project_id}/data_files", timeout=30)
response.raise_for_status()
listing = response.json()

print("data files in one request:", len(listing["data"]))

json_normalize(listing["data"])[["id", "type", "attributes.title"]].head(15)

data files in one request: 57


,id,type,attributes.title
0,6179,data_files,RNA-Seq of Atlantic cod (Gadus morhua) liver t...
1,6178,data_files,Untargeted liver lipid analysis_normalized values
2,4982,data_files,Compound measurements from plasma
3,4981,data_files,Compound measurements from liver
4,4980,data_files,Compound measurements from liver
5,4979,data_files,Compound measurements from liver
6,4978,data_files,Compound measurements from liver
7,4977,data_files,Targeted lipidomics data in MetaboLights repos...
8,2953,data_files,CPM and TCP concentrations in liver and bile
9,2956,data_files,Plasma parameters


## 2.6 Exercises

1. Change `project_id` to another public project (browse https://fairdomhub.org/projects) and list its
   `sops` or `publications`.
2. Extend `related_items` so that it also collects the `description` attribute. Which resource types
   have no description?
3. Fetch `/projects/60/people` and compare the number of rows with the `people` relationship in
   section 2.2. Then fetch the same listing for a project you are *not* a member of but that has
   private items: what is missing, and why?